# Hospital Service Area (HSA) Optimization - v6.0 FINAL
## Unified Scoring System with Mode-Specific Weight Profiles

**Date**: 2026-01-11

**Key Features**:
- **Unified scoring system**: Single formula for all modes with mode-specific weight profiles
- **Pure score-driven optimization**: No artificial facility count constraints
- **Adaptive radius assignment**: Based on urban/rural classification (10km urban, 18km rural)
- **Composite score tracking**: All modes track and export composite scores
- **Overlap removal**: All modes include post-processing to remove HSAs with >80% overlap
- **Configurable parameters**: All key parameters set in notebook (not in .py file)

---

## Five Optimization Modes

### 1. FEWEST (Minimize number of HSAs)
**Goal**: Achieve 90% coverage with minimum facilities  
**Stopping**: Stops immediately when 90% national coverage is reached

**Weight Profile**:
```
coverage: 15.0          # Pure efficiency - maximize coverage per facility
overlap: 0.1            # Minimal overlap penalty
All others: 0.0         # No other factors
```

**Rationale**: Pure efficiency optimization. Selects facilities with highest coverage gain per selection.

---

### 2. FOOTPRINT (Maximize geographic spread)
**Goal**: Achieve 90% coverage with maximum geographic diversity  
**Stopping**: Stops when 90% national coverage is reached

**Weight Profile**:
```
coverage: 0.8           # Moderate coverage preference
climate: 1.2            # PRIMARY DRIVER - rewards climate/geographic diversity
overlap: 0.01           # Minimal overlap penalty (allows dense coverage)
All others: 0.0
```

**Rationale**: Geographic diversity is the primary objective. Climate weight drives selection of facilities across different geographic zones.

---

### 3. DISTANCE (Minimize travel distance)
**Goal**: Achieve 90% coverage while minimizing travel distance  
**Stopping**: Stops when 90% national coverage is reached

**Weight Profile**:
```
coverage: 0.5           # Lower coverage preference
distance: 8.0           # PRIMARY DRIVER - penalizes facilities far from population
overlap: 0.20           # Moderate overlap penalty
climate: 0.05           # Minimal geographic diversity
All others: 0.0
```

**Rationale**: Distance penalty is scaled by coverage gain to prevent overwhelming small coverage gains near target. Encourages selection of centrally-located facilities close to population centers.

---

### 4. GOVERNORATE_TAU_COVERAGE (TAU method per governorate)
**Goal**: Achieve 90% coverage in each governorate independently  
**Stopping**: TAU threshold (90%) met in all governorates

**Weight Profile**: Uses FOOTPRINT weights (balanced approach per governorate)

**Rationale**: Ensures geographic equity by guaranteeing coverage in each administrative region, regardless of population density.

---

### 5. GOVERNORATE_FEWEST (One anchor per governorate + FEWEST)
**Goal**: Guarantee at least 1 HSA per governorate, then minimize total HSAs to reach 90% national coverage  
**Stopping**: 90% national coverage with at least 1 HSA per governorate

**Weight Profile**: Uses FEWEST weights (efficiency-focused)

**Rationale**: Combines administrative equity (1 HSA/governorate minimum) with efficiency (FEWEST scoring). Ensures no governorate is left without a facility.

---

## Scoring System Details

### Base Weights (Configured in Cell 1)

These global weights are applied to normalized 0-1 components, then multiplied by mode-specific profiles:

- `WEIGHT_COVERAGE = 8.0` (population coverage gain)
- `WEIGHT_OVERLAP_PENALTY = 2.0` (redundancy penalty)
- `WEIGHT_CLIMATE = 1.0` (climate/geographic diversity)
- `WEIGHT_PATIENT_VOLUME = 3.0` (facility size - NOT USED in current modes)
- `WEIGHT_COVERAGE_PROGRESS = 1.0` (progress toward target - NOT USED in current modes)
- `WEIGHT_DISTANCE_PENALTY = 3.0` (travel distance penalty - only used in DISTANCE mode)

### Note on Unused Scoring Components

**`patient_volume`**: Set to 0.0 in all current modes because historical patient visits are **not relevant to geographic coverage efficiency**. A facility with 10,000 patients in a dense urban area (already covered by other facilities) contributes less to HSA coverage than a facility with 1,000 patients in an uncovered rural area.

*However*, this component could be useful for **operational optimization modes** not included here:
- **CAPACITY mode**: Select HSAs at facilities with proven operational capacity (established systems, trained staff, supply chains) - useful for emergency preparedness
- **FLOW_CONTINUITY mode**: Minimize disruption by designating HSAs where patients already seek care
- **CONSOLIDATION mode**: Concentrate specialized services at high-volume facilities for economies of scale

**`coverage_progress`**: Set to 0.0 in all current modes because pure optimization goals don't need artificial acceleration toward the target.

*However*, this component could be useful for **constrained optimization modes** not included here:
- **RAPID_DEPLOYMENT mode**: Reach target coverage quickly for pandemic/disaster response (speed over efficiency)
- **FIXED_BUDGET mode**: Maximize coverage with exactly N facilities (ensures all budget is utilized)
- **PROGRESSIVE_EXPANSION mode**: Multi-year implementation plan ensuring smooth, steady progress through coverage plateaus

These components remain in the framework to support future operational/constrained optimization modes without code changes.

### Radius Assignment

1. Compute local population density (10km radius)
2. Classify urban (>1500 people/km²) vs rural
3. Assign base radius:
   - **Urban = 10.0 km**
   - **Rural = 18.0 km**
4. Apply network multiplier: INF=1.0x, NCD=1.0x
5. Compute volume adjustment: -3.0 + 6.0 * percentile (ranging from -3 km to +3 km)
6. Final radius = base radius + volume adjustment

**Result**: Facilities serve catchment areas based on population density and clinical requirements.

### Normalization Details

All scoring components normalized to 0-1 range:
- **coverage_norm** = new_coverage / total_population
- **patient_norm** = facility_volume / max_volume (not used in current modes)
- **overlap_frac** = overlap_population / total_coverage (already 0-1)
- **climate_norm** = 1 / (1 + unique_climate_clusters)
- **progress_norm** = coverage_gain / remaining_to_target (not used in current modes)
- **distance_norm** = coverage_norm × (mean_distance / max_distance) (scaled by coverage gain)

### Stopping Conditions

All modes stop when one of these conditions is met:
1. **Target coverage reached** (90% for all modes)
2. **No more viable facilities** (best score ≤ 0)
3. **All facilities selected** (rare)

Number of facilities **emerges from scoring** - no artificial constraints!

---

## Configuration and Customization

All parameters are set in **Cell 1** and injected into `hsa_optimization.py` at runtime. To customize:

### Adjust Coverage Target
- Modify `TAU_COVERAGE` in Cell 1 (e.g., 0.85 for 85%, 0.95 for 95%)
- Lower values = fewer facilities needed, higher values = more facilities needed

### Adjust Facility Radii
- Modify `URBAN_BASE_RADIUS_KM` and `RURAL_BASE_RADIUS_KM` in Cell 1
- Smaller radii = more facilities needed to achieve coverage
- Larger radii = fewer facilities needed

### Adjust Mode Behavior
- Edit `MODE_WEIGHT_PROFILES` in Cell 1
- **For more facilities**: Increase `climate` weight (FOOTPRINT), decrease `overlap` penalty
- **For fewer facilities**: Increase `coverage` weight (FEWEST), decrease `climate` weight
- **For proximity to population**: Increase `distance` weight (DISTANCE mode)

### Add New Optimization Modes
1. Add new entry to `MODE_WEIGHT_PROFILES` in Cell 1 with custom weights
2. Add mode name to `RUN_OBJECTIVES` dict
3. Mode will automatically use the unified scoring formula with your weights

**All customization is parameter-driven - no code modifications needed!**

---

See cells below for execution and analysis.


In [ ]:
# ============================================================================
# COMPREHENSIVE CONFIGURATION - PARAMETERS ACTUALLY USED
# ============================================================================
# Cleaned configuration with ONLY parameters that are used in the code.
# Removed unused parameters: RNG_SEED, BUFFER_KM, COARSEN_FACTOR, VOLUME_WEIGHT,
# PATIENT_RADIUS_ALPHA, TARGET_HSA_MIN, TARGET_HSA_MAX, CLIMATE_CSV

from pathlib import Path

# --- File Paths ---
DATA_DIR = Path("data")
OUT_DIR = Path("out")
NETWORK = "SYNMODINF"

# --- Geographic Bounds ---
JORDAN_BOUNDS = [34.5, 29.0, 39.5, 33.5]  # [min_lon, min_lat, max_lon, max_lat]

# --- Core Algorithm Parameters ---
TAU_COVERAGE = 0.90                    # Maximum target coverage (used for governorate mode calculation)

# --- Density-Based Classification ---
DENSITY_RADIUS_KM = 10.0               # Radius for density calculation
URBAN_DENSITY_THRESH = 1500.0          # Urban if density > this (people/km²)

# --- Base Radii ---
URBAN_BASE_RADIUS_KM = 10.0            # Base radius for urban facilities
RURAL_BASE_RADIUS_KM = 18.0            # Base radius for rural facilities

# --- Network-Specific Multipliers ---
NETWORK_RADIUS_MULTIPLIERS = {
    'INF': 1.0,        # Infectious disease network
    'SYNINF': 1.0,     # Synthetic infectious disease network
    'SYNMODINF': 1.0,  # Synthetic modeling infectious disease network
    'NCD': 1.0,        # Non-communicable disease network
    'SYNNCD': 1.0,     # Synthetic NCD network
    'SYNMODNCD': 1.0,  # Synthetic modeling NCD network
}

# --- Climate Diversity (applies to ALL objectives) ---
CLIMATE_DIVERSITY_ON = True
CLIMATE_K = 8                          # Number of climate clusters
CLIMATE_MIN_PER_CLUSTER = 0            # Hard floor per climate bin (set 0 to disable)
CLIMATE_CLUSTER_COL = "climate_k"      # Column with precomputed cluster per facility

# --- Scoring Weights (0-10 scale, applied to normalized 0-1 components) ---
# These are the baseline weights, modified by mode-specific profiles below
WEIGHT_COVERAGE = 8.0                  # Population coverage gain (DOMINANT factor)
WEIGHT_OVERLAP_PENALTY = 2.0           # Penalty for redundant coverage
WEIGHT_CLIMATE = 1.0                   # Reward for climate/geographic diversity
WEIGHT_PATIENT_VOLUME = 3.0            # Reward for larger patient volumes
WEIGHT_COVERAGE_PROGRESS = 1.0         # Reward for approaching coverage target
WEIGHT_DISTANCE_PENALTY = 3.0          # Penalty for average travel distance


# --- Post-Processing Thresholds ---
OVERLAP_REMOVAL_THRESHOLD = 0.80           # Remove HSAs with >80% overlap


# --- Mode-Specific Weight Profiles ---
# Each mode applies multipliers to the base weights above
# The optimization stops when target coverage is reached - NO artificial facility count limits

MODE_WEIGHT_PROFILES = {
      'fewest': {
          'coverage': 15.0,          # Pure efficiency - maximize coverage per facility
          'coverage_progress': 0.0,
          'overlap': 0.1,
          'patient_volume': 0.0,     # CRITICAL: Not relevant to HSA coverage efficiency
          'climate': 0.0,
          'distance': 0.0,
      },
      'footprint': {
          'coverage': 0.8,          # Minimal efficiency preference
          'coverage_progress': 0.0,
          'overlap': 0.01,
          'patient_volume': 0.0,
          'climate': 1.2,            # Main driver - geographic diversity
          'distance': 0.0,
      },
      'distance': {
          'coverage': 0.5,
          'coverage_progress': 0.0,
          'overlap': 0.20,
          'patient_volume': 0.0,
          'climate': 0.05,
          'distance': 8.0,
      },
}

# --- Optimization Modes to Run ---
RUN_OBJECTIVES = {
    'fewest': True,                      # Minimize number of HSAs
    'footprint': True,                   # Maximize geographic spread
    'distance': True,                    # Minimize travel distance
    'governorate_tau_coverage': True,    # TAU method per governorate
    'governorate_fewest': True           # One anchor per governorate + FEWEST
}

# --- HSAOptimizer configuration dict ---
config = {
    'pop_path': str(DATA_DIR / 'jor_ppp_2020_UNadj.tif'),  # Population raster (10M version)
    'tau_coverage': 0.90,  # Used for GOVERNORATE_TAU_COVERAGE mode (90% per governorate)
    'coarsen': 4,  # Use FULL resolution (no coarsening) to preserve all 10.2M population. Or 4 to make it 4x faster
}

print("="*80)
print("CONFIGURATION LOADED")
print("="*80)
print(f"Network: {NETWORK}")
print(f"Main modes target coverage: {TAU_COVERAGE*100:.0f}%")
print(f"Governorate mode coverage: {config['tau_coverage']*100:.0f}%")
print(f"Coarsen factor: {config['coarsen']}x (full resolution)")
print(f"Urban base radius: {URBAN_BASE_RADIUS_KM} km")
print(f"Rural base radius: {RURAL_BASE_RADIUS_KM} km")
print(f"Climate diversity: {CLIMATE_DIVERSITY_ON}")
print(f"Optimization is purely score-driven (no facility count limits)")
print("="*80)

In [ ]:
# Generate diagnosis counts for the selected network
import sys
import subprocess

# Diagnosis counts path resolution: mode-aware when available, safe fallback when not.
mode_suffix = f"_{HSA_MODE}" if 'HSA_MODE' in globals() and HSA_MODE else ""
diag_candidates = [OUT_DIR / f'{NETWORK}_diagnosis_counts_pivot.csv']
if mode_suffix:
    diag_candidates.append(OUT_DIR / f'{NETWORK}{mode_suffix}_diagnosis_counts_pivot.csv')
diag_candidates.append(OUT_DIR / f'{NETWORK}_footprint_diagnosis_counts_pivot.csv')

diag_csv = next((c for c in diag_candidates if c.exists()), None)
groups_csv = DATA_DIR / f'{NETWORK}_groups_of_diagnoses.csv'
patients_csv = DATA_DIR / f'{NETWORK}_patient_visits.csv'

if not groups_csv.exists():
    raise FileNotFoundError(f'Diagnosis groups not found: {groups_csv}')
if not patients_csv.exists():
    raise FileNotFoundError(f'Patient visits not found: {patients_csv}')

print(f'Preparing diagnosis counts for {NETWORK}...')
result = subprocess.run([sys.executable, 'generate_diagnosis_counts_v2.py', '--networks', NETWORK], check=False)
if result.returncode != 0:
    raise RuntimeError(f'Diagnosis counts generation failed (code {result.returncode}).')
diag_csv = next((c for c in diag_candidates if c.exists()), diag_csv)
if diag_csv is None or not diag_csv.exists():
    raise FileNotFoundError(f'Diagnosis counts not found after generation. Checked: {diag_candidates}')
print(f'✓ Diagnosis counts ready: {diag_csv}')


In [ ]:
%load_ext autoreload
%autoreload 2


# ============================================================================
# IMPORTS AND PARAMETER INJECTION
# ============================================================================

# Import libraries
import sys
from pathlib import Path
import geopandas as gpd
import pandas as pd

# Import optimization module
import hsa_optimization
from hsa_optimization import HSAOptimizer

# ============================================================================
# INJECT CONFIGURATION INTO HSA_OPTIMIZATION MODULE
# ============================================================================
# The hsa_optimization module no longer defines its own parameters.
# Instead, it expects them to be injected from the notebook.
# This makes all parameters visible and editable in one place (Cell 1 above).

param_names = [
      'DATA_DIR', 'OUT_DIR', 'NETWORK',
      'TAU_COVERAGE',
      'DENSITY_RADIUS_KM', 'URBAN_DENSITY_THRESH',
      'URBAN_BASE_RADIUS_KM', 'RURAL_BASE_RADIUS_KM',
      'NETWORK_RADIUS_MULTIPLIERS',
      'CLIMATE_DIVERSITY_ON', 'CLIMATE_K', 'CLIMATE_MIN_PER_CLUSTER',
      'CLIMATE_CLUSTER_COL',
      'WEIGHT_COVERAGE', 'WEIGHT_OVERLAP_PENALTY', 'WEIGHT_CLIMATE',
      'WEIGHT_PATIENT_VOLUME', 'WEIGHT_COVERAGE_PROGRESS', 'WEIGHT_DISTANCE_PENALTY',
      'MODE_WEIGHT_PROFILES',
      'JORDAN_BOUNDS', 'OVERLAP_REMOVAL_THRESHOLD',  # Added new parameters
]


print("Injecting parameters into hsa_optimization module...")
injected_count = 0
for param in param_names:
    if param in globals():
        setattr(hsa_optimization, param, globals()[param])
        injected_count += 1
    else:
        print(f"  WARNING: Parameter {param} not found in globals()")

print("="*80)
print("IMPORTS AND PARAMETER INJECTION COMPLETE")
print("="*80)
print(f"Parameters injected: {injected_count}/{len(param_names)}")
print("="*80)

## Run Optimizations

Runs all enabled optimization modes (configured in Cell 1 via `RUN_OBJECTIVES`).

Results saved to GeoJSON files in the `out/` directory.

In [ ]:
# Initialize optimizer
optimizer = HSAOptimizer(config)



# Load facilities WITH CLIMATE DATA from CSV
climate_csv = OUT_DIR / f'{NETWORK}_Facilities_Climate_Features_with_clusters.csv'

if not climate_csv.exists():
    raise FileNotFoundError(f"Climate CSV not found: {climate_csv}")

# Read climate CSV
climate_df = pd.read_csv(climate_csv)

# Rename FacilityName to HealthFacility (what the optimizer expects)
if 'FacilityName' in climate_df.columns:
    climate_df['HealthFacility'] = climate_df['FacilityName']
elif 'HealthFacility' not in climate_df.columns:
    raise ValueError("Climate CSV must have either 'FacilityName' or 'HealthFacility' column")

# NORMALIZE WHITESPACE: Collapse multiple spaces to single space
climate_df['HealthFacility'] = climate_df['HealthFacility'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Load diagnosis counts (for Total column)
# Diagnosis counts path resolution: mode-aware when available, safe fallback when not.
mode_suffix = f"_{HSA_MODE}" if 'HSA_MODE' in globals() and HSA_MODE else ""
diag_candidates = [OUT_DIR / f'{NETWORK}_diagnosis_counts_pivot.csv']
if mode_suffix:
    diag_candidates.append(OUT_DIR / f'{NETWORK}{mode_suffix}_diagnosis_counts_pivot.csv')
diag_candidates.append(OUT_DIR / f'{NETWORK}_footprint_diagnosis_counts_pivot.csv')

diagnosis_csv = next((c for c in diag_candidates if c.exists()), diag_candidates[0])
if not diagnosis_csv.exists():
    raise FileNotFoundError(f"Diagnosis counts not found: {diagnosis_csv}")

diagnosis_df = pd.read_csv(diagnosis_csv)

# NORMALIZE WHITESPACE in diagnosis data as well
diagnosis_df['healthfacility'] = diagnosis_df['healthfacility'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Merge climate data with diagnosis counts on HealthFacility
merged_df = climate_df.merge(
    diagnosis_df[['healthfacility', 'total_diagnoses']],
    left_on='HealthFacility',
    right_on='healthfacility',
    how='left'
)

# Rename total_diagnoses to Total (what the optimizer expects)
merged_df['Total'] = merged_df['total_diagnoses']

# Check for missing values
missing_total = merged_df['Total'].isna().sum()
if missing_total > 0:
    print(f"WARNING: {missing_total} facilities missing diagnosis counts")
    print(f"  Missing facilities:")
    missing_facilities = merged_df[merged_df['Total'].isna()]['HealthFacility'].tolist()
    for fac in missing_facilities[:10]:  # Show first 10
        print(f"    - {fac}")
    if len(missing_facilities) > 10:
        print(f"    ... and {len(missing_facilities) - 10} more")
    # Fill with 0
    merged_df['Total'] = merged_df['Total'].fillna(0)

# Create GeoDataFrame from lat/lon
facilities = gpd.GeoDataFrame(
    merged_df,
    geometry=gpd.points_from_xy(merged_df['lon'], merged_df['lat']),
    crs='EPSG:4326'
)

# Reproject to UTM for distance calculations (Jordan is in UTM zone 36N/37N)
# Using a common Jordan projection: EPSG:32637 (UTM 37N)
facilities = facilities.to_crs('EPSG:32637')

print(f"\nMerged climate + diagnosis data:")
print(f"  Facilities with climate data: {len(climate_df)}")
print(f"  Facilities with diagnosis counts: {len(diagnosis_df)}")
print(f"  Facilities after merge: {len(facilities)}")
print(f"  Facilities with Total column: {facilities['Total'].notna().sum()}")
print(f"  Facilities with Total > 0: {(facilities['Total'] > 0).sum()}")
print(f"  Facilities with HealthFacility column: {'HealthFacility' in facilities.columns}")


print(f"Loaded {len(facilities)} {NETWORK} facilities ")
if 'climate_k' in facilities.columns:
    print(f"  climate_k column present: YES")
    print(f"  climate_k values: {sorted(facilities['climate_k'].dropna().unique())}")
else:
    print(f"  climate_k column present: NO (climate diversity scoring will be disabled!)")

# Load governorates for governorate modes
gov_file = DATA_DIR / 'jordan_governorates.gpkg'
governorates = gpd.read_file(gov_file) if (DATA_DIR / 'jordan_governorates.gpkg').exists() else None

# Store results for all objectives
all_results = {}

# Run optimizations for selected objectives
for objective in ['fewest', 'footprint', 'distance', 'governorate_tau_coverage', 'governorate_fewest']:
    if objective not in RUN_OBJECTIVES or not RUN_OBJECTIVES[objective]:
        print(f"\nSkipping {objective.upper()} optimization (disabled)")
        continue
    
    print(f"\n{'='*80}")
    print(f"Running {objective.upper()} optimization...")
    print('='*80)
    
    # Run optimization
    if objective in ['governorate_tau_coverage', 'governorate_fewest']:
        if governorates is None:
            print("  ERROR: Governorates file not found, skipping")
            continue
        result = optimizer.optimize(facilities, objective=objective, network_type=NETWORK, governorates_gdf=governorates)
    else:
        result = optimizer.optimize(facilities, objective=objective, network_type=NETWORK)

    
    selected_facilities = result['facilities']
    coverage_pct = result['coverage_pct']
    
    all_results[objective] = {
        'result': result,
        'facilities': selected_facilities,
        'coverage': coverage_pct
    }
    
    print(f"\n{'='*80}")
    print(f"{objective.upper()} OPTIMIZATION COMPLETE")
    print('='*80)
    print(f"Selected: {len(selected_facilities)} HSAs")
    print(f"Coverage: {coverage_pct:.1f}%")
    
    # ========================================================================
    # SAVE RESULTS TO GEOJSON (v2 files)
    # ========================================================================
    print(f"\nSaving results to GeoJSON...")
    
    # Add metadata
    selected_with_meta = selected_facilities.copy().reset_index(drop=True)
    selected_with_meta['network_type'] = NETWORK
    selected_with_meta['optimization_mode'] = objective

    # Create HSA boundary polygons from selected facilities
    hsas_polygons = hsa_optimization.create_hsa_polygons(selected_with_meta).reset_index(drop=True)
    hsas_gdf = selected_with_meta.copy()
    hsas_gdf['geometry'] = hsas_polygons.geometry.values
    
    # Clip HSA circles to WorldPop populated cells (removes uninhabited desert)
    print(f"  Clipping HSAs to populated WorldPop cells...")
    # Use constrained WorldPop for clipping: only assigns population where
    # buildings are detected, so true desert = 0 (UNadj spreads population
    # to all land cells including desert, preventing desert carve-outs)
    constrained_pop_path = DATA_DIR / 'jor_ppp_2020_constrained.tif'
    hsas_gdf = hsa_optimization.clip_hsas_to_population(
        hsas_gdf, str(constrained_pop_path), min_pop=0.0, coarsen=2,
        min_patch_km2=0.5, smooth_m=500
    )

    # Save to v2.geojson file
    output_file = OUT_DIR / f'{NETWORK}_{objective}_hsas_v2.geojson'
    hsas_gdf.to_file(output_file, driver='GeoJSON')
    
    # Verify saved file
    reloaded = gpd.read_file(output_file)
    print(f"  Saved: {output_file.name}")
    print(f"  Columns in saved file: {len(reloaded.columns)}")
    print(f"  climate_k preserved: {'climate_k' in reloaded.columns}")
    
    if 'service_radius_km' in reloaded.columns:
        max_radius = reloaded['service_radius_km'].max()
        print(f"  Max service_radius_km: {max_radius:.1f} km (should be <= 30.0)")
        if max_radius > 30.0:
            print(f"  WARNING: Radius exceeds 30 km limit!")
    
    print(f"  File verified successfully")

print(f"\n\n{'='*80}")
print("ALL OPTIMIZATIONS COMPLETE")
print('='*80)
for obj, data in all_results.items():
    print(f"  {obj.upper():26s}: {len(data['facilities']):3d} HSAs, {data['coverage']:.1f}% coverage")


## Analyze Results

In [ ]:
# ============================================================================
# SECTIONS 3-8: ANALYZE ALL OPTIMIZATION MODES
# ============================================================================

from hsa_objective_analysis import analyze_all_modes

# Analyze all modes with complete suite of visualizations and statistics
analyze_all_modes(all_results)

In [ ]:
# ============================================================================
# HELPER: Normalize composite scores for cross-mode comparison
# ============================================================================

import numpy as np
import pandas as pd

def _normalize_scores_per_mode(all_results):
    summary_rows = []
    for mode, data in all_results.items():
        df = data.get('facilities')
        if df is None or 'composite_score' not in df.columns:
            continue
        scores = df['composite_score'].astype(float)
        min_s = scores.min()
        max_s = scores.max()
        if max_s > min_s:
            norm = (scores - min_s) / (max_s - min_s)
        else:
            norm = scores * 0.0
        std = scores.std(ddof=0)
        if std > 0:
            z = (scores - scores.mean()) / std
        else:
            z = scores * 0.0
        df = df.copy()
        df['composite_score_norm'] = norm
        df['composite_score_z'] = z
        data['facilities'] = df
        summary_rows.append({
            'mode': mode,
            'min': min_s,
            'median': scores.median(),
            'mean': scores.mean(),
            'max': max_s,
            'mean_norm': norm.mean(),
            'median_norm': float(np.median(norm)),
        })
    if not summary_rows:
        return pd.DataFrame()
    return pd.DataFrame(summary_rows).sort_values('mode')

score_summary = _normalize_scores_per_mode(all_results)
if not score_summary.empty:
    display(score_summary)
    print("Normalized scores added per mode: composite_score_norm (0-1) and composite_score_z (z-score).")
else:
    print("No composite_score column found in all_results.")


In [ ]:
# ============================================================================
# CROSS-MODE COMPARISON: Facility Counts and HSA Volumes
# ============================================================================

from geopandas import sjoin
import rasterio
from rasterio.mask import mask as raster_mask
import pandas as pd


# Load all facilities once from climate CSV + diagnosis counts
climate_csv = OUT_DIR / f'{NETWORK}_Facilities_Climate_Features_with_clusters.csv'
# Diagnosis counts path resolution: mode-aware when available, safe fallback when not.
mode_suffix = f"_{HSA_MODE}" if 'HSA_MODE' in globals() and HSA_MODE else ""
diag_candidates = [OUT_DIR / f'{NETWORK}_diagnosis_counts_pivot.csv']
if mode_suffix:
    diag_candidates.append(OUT_DIR / f'{NETWORK}{mode_suffix}_diagnosis_counts_pivot.csv')
diag_candidates.append(OUT_DIR / f'{NETWORK}_footprint_diagnosis_counts_pivot.csv')

diagnosis_csv = next((c for c in diag_candidates if c.exists()), diag_candidates[0])

climate_df = pd.read_csv(climate_csv)
diagnosis_df = pd.read_csv(diagnosis_csv)

# Rename FacilityName to HealthFacility
if 'FacilityName' in climate_df.columns:
    climate_df['HealthFacility'] = climate_df['FacilityName']

# NORMALIZE WHITESPACE: Collapse multiple spaces to single space
climate_df['HealthFacility'] = climate_df['HealthFacility'].str.replace(r'\s+', ' ', regex=True).str.strip()
diagnosis_df['healthfacility'] = diagnosis_df['healthfacility'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Merge
merged_df = climate_df.merge(
    diagnosis_df[['healthfacility', 'total_diagnoses']],
    left_on='HealthFacility',
    right_on='healthfacility',
    how='left'
)
merged_df['Total'] = merged_df['total_diagnoses'].fillna(0)

all_facilities = gpd.GeoDataFrame(
    merged_df,
    geometry=gpd.points_from_xy(merged_df['lon'], merged_df['lat']),
    crs='EPSG:4326'
).to_crs('EPSG:32637')





# Compare all modes
comparison_results = []

for mode in ['fewest', 'footprint', 'distance', 'governorate_tau_coverage', 'governorate_fewest']:
    if mode not in all_results:
        continue

    # Get HSAs and ensure they're in EPSG:32637
    final_hsas_raw = all_results[mode]['facilities']

    # Build HSA polygons for analysis
    hsas_polygons = hsa_optimization.create_hsa_polygons(final_hsas_raw)
    final_hsas_raw = final_hsas_raw.copy()
    final_hsas_raw['geometry'] = hsas_polygons.geometry

    # Ensure CRS is set correctly (in case it's lost during optimization)
    if final_hsas_raw.crs is None:
        final_hsas_utm = final_hsas_raw.set_crs('EPSG:32637')
    elif final_hsas_raw.crs != 'EPSG:32637':
        final_hsas_utm = final_hsas_raw.to_crs('EPSG:32637')
    else:
        final_hsas_utm = final_hsas_raw

    N_ANCHORS = len(final_hsas_utm)

    print(f"\n{'='*80}")
    print(f"{mode.upper()} - DETAILED HSA ANALYSIS ({N_ANCHORS} HSAs)")
    print('='*80)

    # Calculate detailed HSA statistics
    details = []

    for idx, row in final_hsas_utm.iterrows():
        # Get anchor facility volume
        anchor_volume = pd.to_numeric(row['Total'], errors='coerce')

        # Get HSA center point (in UTM)
        center_point_utm = row.geometry
        # Flag empty geometries
        if center_point_utm is None or getattr(center_point_utm, 'is_empty', False):
            fac_name = row.get('HealthFacility', row.get('FacilityName'))
            print(f"  WARNING: Empty HSA geometry for {fac_name}")
            continue
        if center_point_utm is None or getattr(center_point_utm, 'is_empty', False):
            fac_name = row.get('HealthFacility', row.get('FacilityName'))
            match = all_facilities[all_facilities['HealthFacility'] == fac_name]
            if match.empty:
                print(f"  WARNING: Missing geometry for {fac_name}; skipping")
                continue
            center_point_utm = match.geometry.iloc[0]

        # Get radius
        radius_km = row.get('service_radius_km', row.get('initial_radius_km', 15.0))

        # Create circle in UTM (meters) for facility counting
        radius_meters = radius_km * 1000
        hsa_circle_utm = center_point_utm.buffer(radius_meters)

        # Find ALL facilities within this HSA circle
        # Use explicit CRS to avoid any mismatch
        hsa_temp = gpd.GeoDataFrame([{'geometry': hsa_circle_utm}], crs='EPSG:32637')
        facilities_in_hsa = sjoin(all_facilities, hsa_temp, how='inner', predicate='intersects')

        # Calculate total HSA volume
        total_hsa_volume = 0
        for fac_idx, fac_row in facilities_in_hsa.iterrows():
            fac_vol = pd.to_numeric(fac_row['Total'], errors='coerce')
            if not pd.isna(fac_vol):
                total_hsa_volume += fac_vol

        # For population: convert circle to EPSG:4326 for raster
        hsa_circle_latlon = gpd.GeoDataFrame(
            [{'geometry': hsa_circle_utm}],
            crs='EPSG:32637'
        ).to_crs('EPSG:4326').geometry.iloc[0]

        # Calculate population
        try:
            with rasterio.open(config['pop_path']) as src:
                out_image, out_transform = raster_mask(src, [hsa_circle_latlon], crop=True, nodata=0)
                total_pop = float(out_image.sum())
        except Exception as e:
            total_pop = 0

        # Append details with ALL required columns
        details.append({
            'Mode': mode.upper(),
            'HSA_Anchor': row['FacilityName'],
            'Anchor_Patient_Volume': anchor_volume,
            'Num_Facilities_in_HSA': len(facilities_in_hsa),
            'Total_HSA_Volume': total_hsa_volume,
            'HSA_Population': total_pop,
            'Service_Radius_km': row.get('service_radius_km', row.get('initial_radius_km', 0)),
            'Composite_Score': row.get('composite_score', 0)
        })

    # Create DataFrame
    detail_df = pd.DataFrame(details)

    # Define display columns
    display_cols = ['HSA_Anchor', 'Anchor_Patient_Volume', 'Num_Facilities_in_HSA',
                    'Total_HSA_Volume', 'HSA_Population', 'Service_Radius_km', 'Composite_Score']

    # Show ALL facilities (not .head(15))
    print(detail_df[display_cols].to_string(index=False))

    # Summary statistics
    print(f"\nSummary Statistics for {mode.upper()}:")
    print(f"  Total HSAs: {len(detail_df)}")
    print(f"  Avg facilities per HSA: {detail_df['Num_Facilities_in_HSA'].mean():.1f}")
    print(f"  Total facilities covered: {detail_df['Num_Facilities_in_HSA'].sum()}")
    print(f"  Avg HSA volume: {detail_df['Total_HSA_Volume'].mean():,.0f}")
    print(f"  Total HSA volume: {detail_df['Total_HSA_Volume'].sum():,.0f}")

    # Add to comparison
    comparison_results.append({
        'Mode': mode.upper(),
        'Num_HSAs': N_ANCHORS,
        'Avg_Facilities_per_HSA': detail_df['Num_Facilities_in_HSA'].mean(),
        'Total_Facilities_Covered': detail_df['Num_Facilities_in_HSA'].sum(),
        'Avg_HSA_Volume': detail_df['Total_HSA_Volume'].mean(),
        'Total_HSA_Volume': detail_df['Total_HSA_Volume'].sum()
    })

# Cross-mode comparison table
print(f"\n\n{'='*80}")
print("CROSS-MODE COMPARISON SUMMARY")
print('='*80)
comparison_df = pd.DataFrame(comparison_results)
print(comparison_df.to_string(index=False))


## Generate HSA Maps

Create HSA boundary polygons and professional maps for all optimization results.

In [ ]:
# Load Jordan country boundary for clipping
country_file = DATA_DIR / 'jordan_admin0.gpkg'  # Adjust filename if needed
if not country_file.exists():
    country_file = DATA_DIR / 'jordan_boundary.gpkg'
if not country_file.exists():
    country_file = DATA_DIR / 'jordan_governorates.gpkg'  # Use governorates as fallback

if country_file.exists():
    country_boundary = gpd.read_file(country_file)
    if 'jordan_governorates' in str(country_file):
        # Dissolve governorates to create country boundary
        country_boundary = country_boundary.dissolve()
    print(f"Country boundary loaded from: {country_file.name}")
    print(f"  CRS: {country_boundary.crs}")
else:
    print("WARNING: Country boundary file not found. Maps will not be clipped.")
    country_boundary = None

In [ ]:
# ============================================================================
# CREATE MAPS FOR ALL MODES
# ============================================================================

# FORCE FRESH IMPORT - clear any cached modules
import sys
if 'hsa_mapping' in sys.modules:
    del sys.modules["hsa_mapping"]
if 'hsa_mapping_working' in sys.modules:
    del sys.modules["hsa_mapping_working"]

# Import the WORKING mapping module (extracted from create_geopandas_maps.py)
from hsa_mapping_working import create_all_hsa_maps, save_hsa_geopackages

print("Using hsa_mapping_working module (extracted from create_geopandas_maps.py)")
print("This version has been verified to produce correct maps with:")
print("  - Circular HSAs (not stretched)")
print("  - Visible population grid")

# Create all maps with proper layer ordering
create_all_hsa_maps(
    all_results=all_results,
    network=NETWORK,
    data_dir=DATA_DIR,
    out_dir=OUT_DIR
)

# Save GeoPackages (one per mode, all map layers bundled)
save_hsa_geopackages(
    all_results=all_results,
    network=NETWORK,
    data_dir=DATA_DIR,
    out_dir=OUT_DIR
)

## Generate HSA Detail Tables

In [ ]:
# ============================================================================
# HSA DETAIL TABLES - ALL MODES, ALL HSAS
# ============================================================================

from geopandas import sjoin
import rasterio
from rasterio.mask import mask as raster_mask
import pandas as pd



# Load all facilities once from climate CSV + diagnosis counts
climate_csv = OUT_DIR / f'{NETWORK}_Facilities_Climate_Features_with_clusters.csv'
# Diagnosis counts path resolution: mode-aware when available, safe fallback when not.
mode_suffix = f"_{HSA_MODE}" if 'HSA_MODE' in globals() and HSA_MODE else ""
diag_candidates = [OUT_DIR / f'{NETWORK}_diagnosis_counts_pivot.csv']
if mode_suffix:
    diag_candidates.append(OUT_DIR / f'{NETWORK}{mode_suffix}_diagnosis_counts_pivot.csv')
diag_candidates.append(OUT_DIR / f'{NETWORK}_footprint_diagnosis_counts_pivot.csv')

diagnosis_csv = next((c for c in diag_candidates if c.exists()), diag_candidates[0])

climate_df = pd.read_csv(climate_csv)
diagnosis_df = pd.read_csv(diagnosis_csv)

# Rename FacilityName to HealthFacility
if 'FacilityName' in climate_df.columns:
    climate_df['HealthFacility'] = climate_df['FacilityName']

# NORMALIZE WHITESPACE: Collapse multiple spaces to single space
climate_df['HealthFacility'] = climate_df['HealthFacility'].str.replace(r'\s+', ' ', regex=True).str.strip()
diagnosis_df['healthfacility'] = diagnosis_df['healthfacility'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Merge
merged_df = climate_df.merge(
    diagnosis_df[['healthfacility', 'total_diagnoses']],
    left_on='HealthFacility',
    right_on='healthfacility',
    how='left'
)
merged_df['Total'] = merged_df['total_diagnoses'].fillna(0)

all_facilities = gpd.GeoDataFrame(
    merged_df,
    geometry=gpd.points_from_xy(merged_df['lon'], merged_df['lat']),
    crs='EPSG:4326'
).to_crs('EPSG:32637')





modes_to_process = {
    'fewest': 'FEWEST',
    'footprint': 'FOOTPRINT',
    'distance': 'DISTANCE',
    'governorate_tau_coverage': 'GOVERNORATE TAU COVERAGE',
    'governorate_fewest': 'GOVERNORATE FEWEST'
}


for mode_name, mode_label in modes_to_process.items():
    hsas_file = OUT_DIR / f'{NETWORK}_{mode_name}_hsas_v2.geojson'

    if not hsas_file.exists():
        print(f"\nSkipping {mode_label} - file not found")
        continue

    # Load HSAs
    hsas_points = gpd.read_file(hsas_file)
    hsas_points_utm = hsas_points.to_crs(all_facilities.crs)

    print(f"\n{'='*80}")
    print(f"{mode_label} - ALL {len(hsas_points)} HSAs")
    print('='*80)

    # Calculate details for ALL HSAs
    details = []

    for idx, row in hsas_points_utm.iterrows():
        # Get anchor facility volume
        anchor_volume = pd.to_numeric(row['Total'], errors='coerce')
  
        # Get HSA center point (in UTM)
        center_point_utm = row.geometry
  
        # Get radius
        radius_km = row.get('service_radius_km', row.get('initial_radius_km', 15.0))
  
        # Create circle in UTM (meters) for facility counting
        radius_meters = radius_km * 1000
        hsa_circle_utm = center_point_utm.buffer(radius_meters)
  
        # Find ALL facilities within this HSA circle
        hsa_temp = gpd.GeoDataFrame([{'geometry': hsa_circle_utm}], crs=hsas_points_utm.crs)
        facilities_in_hsa = sjoin(all_facilities, hsa_temp, how='inner', predicate='intersects')
  
        # Calculate total HSA volume
        total_hsa_volume = 0
        for fac_idx, fac_row in facilities_in_hsa.iterrows():
            fac_vol = pd.to_numeric(fac_row['Total'], errors='coerce')
            if not pd.isna(fac_vol):
                total_hsa_volume += fac_vol
  
        # For population: convert circle to EPSG:4326 for raster
        hsa_circle_latlon = gpd.GeoDataFrame(
            [{'geometry': hsa_circle_utm}],
            crs=hsas_points_utm.crs
        ).to_crs('EPSG:4326').geometry.iloc[0]
  
        # Calculate population
        try:
            with rasterio.open(config['pop_path']) as src:
                out_image, out_transform = raster_mask(src, [hsa_circle_latlon], crop=True, nodata=0)
                total_pop = float(out_image.sum())
        except Exception as e:
            total_pop = 0
  
        details.append({
            'HSA_Anchor': row['FacilityName'],
            'Anchor_Volume': anchor_volume,
            'Num_Facilities_in_HSA': len(facilities_in_hsa),
            'Total_HSA_Volume': total_hsa_volume,
            'HSA_Population': total_pop,
            'Service_Radius_km': radius_km,
            'Composite_Score': row.get('composite_score', 0)
        })
        # REMOVE ANY PRINT STATEMENTS HERE!

    # ONLY PRINT AFTER THE LOOP COMPLETES
    detail_df = pd.DataFrame(details)
    print(detail_df.to_string(index=False))

    # Summary statistics
    print(f"\nSummary Statistics:")
    print(f"  Total HSAs: {len(detail_df)}")
    print(f"  Total facilities across all HSAs: {detail_df['Num_Facilities_in_HSA'].sum()}")
    print(f"  Total HSA volume: {detail_df['Total_HSA_Volume'].sum():,.0f}")
    print(f"  Total population covered: {detail_df['HSA_Population'].sum():,.0f}")

print("\n" + "="*80)
print("HSA DETAIL TABLES COMPLETE")
print("="*80)


In [ ]:
# AUTO_NOTEBOOK_SUMMARY_V1
from pathlib import Path
from datetime import datetime
import os
import json

NOTEBOOK_NAME = "HSA_v6_FINAL"
NETWORK = globals().get('NETWORK', os.environ.get('NETWORK', 'INF'))
HSA_MODE = globals().get('HSA_MODE', os.environ.get('HSA_MODE', ''))

suffix = f"{NETWORK}_{HSA_MODE}" if HSA_MODE else f"{NETWORK}"

out_root = Path('out')
summary_dir = out_root / 'textresults'
summary_dir.mkdir(parents=True, exist_ok=True)
summary_path = summary_dir / f"{NOTEBOOK_NAME}_{suffix}_results.md"

files = [p for p in out_root.rglob('*') if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.md', '.txt', '.png', '.pdf', '.geojson', '.parquet'}]
files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
important = files[:120]

NL = "\n"

blocks = []
blocks.append(f"# Notebook Results: {NOTEBOOK_NAME}")

meta = [
    f"- Generated: {datetime.now().isoformat(timespec='seconds')}",
    f"- Network: {NETWORK}",
]
if HSA_MODE:
    meta.append(f"- HSA mode: {HSA_MODE}")
blocks.append(NL.join(meta))

file_lines = ['## Important Output Files', '']
for p in important:
    file_lines.append(f"- `{p}`")
blocks.append(NL.join(file_lines))

nb_path = Path(f"{NOTEBOOK_NAME}.ipynb")
if nb_path.exists():
    try:
        nb_data = json.loads(nb_path.read_text())
        blocks.append('## Displayed Cell Outputs')

        for idx, cell in enumerate(nb_data.get('cells', []), start=1):
            if cell.get('cell_type') != 'code':
                continue
            outputs = cell.get('outputs', []) or []
            if not outputs:
                continue

            blocks.append(f"### Cell {idx}")

            for out in outputs:
                otype = out.get('output_type')

                if otype == 'stream':
                    text = ''.join(out.get('text', [])) if isinstance(out.get('text', []), list) else str(out.get('text', ''))
                    text = text.rstrip()
                    if text:
                        blocks.append("```text" + NL + text + NL + "```")

                elif otype in ('execute_result', 'display_data'):
                    data = out.get('data', {})
                    if 'text/markdown' in data:
                        md = ''.join(data['text/markdown']) if isinstance(data['text/markdown'], list) else str(data['text/markdown'])
                        md = md.rstrip()
                        if md:
                            blocks.append(md)
                    elif 'text/plain' in data:
                        txt = ''.join(data['text/plain']) if isinstance(data['text/plain'], list) else str(data['text/plain'])
                        txt = txt.rstrip()
                        if txt:
                            blocks.append("```text" + NL + txt + NL + "```")
                    elif 'text/html' in data:
                        html = ''.join(data['text/html']) if isinstance(data['text/html'], list) else str(data['text/html'])
                        html = html.rstrip()
                        if html:
                            blocks.append("```html" + NL + html + NL + "```")
                    elif 'image/png' in data or 'image/jpeg' in data:
                        blocks.append('_[Image output omitted in text summary]_')

                elif otype == 'error':
                    tb = out.get('traceback', []) or []
                    if tb:
                        err = NL.join(str(t) for t in tb)
                    else:
                        err = f"{out.get('ename', 'Error')}: {out.get('evalue', '')}"
                    blocks.append("```text" + NL + err + NL + "```")

    except Exception as e:
        blocks.append('## Displayed Cell Outputs')
        blocks.append(f"Could not parse notebook outputs: {e}")

summary = (NL + NL).join(b for b in blocks if b and str(b).strip()) + NL
summary_path.write_text(summary)
print(f"Saved notebook summary: {summary_path}")
